# Cuaderno 1 — Universo de estudio: empresas tecnológicas de crecimiento

**Tesis:** Predicción del Punto de Equilibrio de Flujo de Caja en Empresas de Crecimiento del Sector Tecnológico

**Universidad EAFIT · 2026**

---

## Objetivo

Definir el universo de estudio y construir el dataset histórico de variables
financieras para las empresas tecnológicas de pequeña y mediana capitalización
que conforman la muestra de esta investigación.

---

## Universo de estudio: constituyentes tecnológicos del Russell 2500

La tesis estudia empresas de crecimiento de pequeña y mediana capitalización
(*small y mid-cap*) del sector tecnológico. El universo se define siguiendo la
metodología del **Russell 2500** — el índice de referencia institucional para
el segmento *small/mid-cap* del mercado accionario estadounidense (FTSE Russell,
2026) —, delimitado exclusivamente a las empresas cuya actividad principal
corresponde a la industria tecnológica.

### Delimitación sectorial

El universo se acota mediante los códigos de Clasificación Industrial Estándar
(SIC) correspondientes a la industria tecnológica: software empresarial y de
consumo, semiconductores y componentes, procesamiento y servicios de datos,
diseño de sistemas integrados, y equipos de comunicación y electrónica.

Esta delimitación responde a la observación metodológica del asesor de tesis:
los ciclos de maduración e inversión de las empresas tecnológicas difieren
sustancialmente de los de otros sectores, lo que justifica un modelo
predictivo específico.

---

## Fuente de datos: SEC EDGAR

Se utilizan formularios 10-Q (trimestral, no auditado) y 10-K (anual, auditado).

---

## Ajuste al protocolo de validación temporal

El entrenamiento inicial cubrirá 2008–2017, con el primer año de evaluación en
2018, generando 7 *folds* de validación walk-forward (2018–2024).

## 1. Configuración inicial

In [3]:
import pandas as pd

# --- Cargar las variables financieras crudas (ya incluyen ticker y sector) ---
variables_crudas = pd.read_csv(
    "../datos/crudos/variables_financieras_sec.csv.gz",
    compression="gzip"
)

# --- Delimitar el universo tecnológico mediante la columna 'sector' ---
segmentos_tecnologicos = [
    "software", "computer", "semiconductor", "internet", "data processing",
    "electronic", "communications equipment", "computer integrated",
    "computer peripheral", "computer communications", "computer programming"
]

mask_tech = variables_crudas["sector"].str.lower().str.contains(
    "|".join(segmentos_tecnologicos), na=False
)
variables_tech = variables_crudas[mask_tech].copy()

print(f"Empresas únicas en el universo tecnológico: {variables_tech['ticker'].nunique()}")
print(f"Registros financieros del universo tecnológico: {len(variables_tech)}")
print(f"\nDistribución por segmento (sector):")
print(variables_tech["sector"].value_counts())

# --- Guardar el dataset filtrado ---
variables_tech.to_csv("../datos/crudos/variables_financieras_tech.csv.gz",
                       compression="gzip", index=False)
print(f"\nArchivo guardado: variables_financieras_tech.csv.gz")

Empresas únicas en el universo tecnológico: 271
Registros financieros del universo tecnológico: 695443

Distribución por segmento (sector):
sector
Services-Prepackaged Software                                  192631
Semiconductors & Related Devices                               136581
Services-Computer Processing & Data Preparation                 66718
Services-Computer Integrated Systems Design                     54469
Services-Computer Programming, Data Processing, Etc.            45355
Electronic Components, NEC                                      22934
Radio & Tv Broadcasting & Communications Equipment              22863
Communications Equipment, NEC                                   21735
Wholesale-Computers & Peripheral Equipment & Software           20424
Electronic Components & Accessories                             19288
Calculating & Accounting Machines (No Electronic Computers)     18179
Computer Communications Equipment                               16550
Computer Peri

In [4]:
# --- Excluir categorías que no son desarrollo/fabricación tecnológica pura ---
categorias_excluidas = [
    "Retail-Computer & Computer Software Stores",
    "Wholesale-Computers & Peripheral Equipment & Software",
    "Wholesale-Electronic Parts & Equipment, NEC",
    "Calculating & Accounting Machines (No Electronic Computers)"
]

variables_tech = variables_tech[~variables_tech["sector"].isin(categorias_excluidas)].copy()

print(f"Empresas únicas tras exclusión: {variables_tech['ticker'].nunique()}")
print(f"Registros financieros tras exclusión: {len(variables_tech)}")

variables_tech.to_csv("../datos/crudos/variables_financieras_tech.csv.gz",
                       compression="gzip", index=False)
print(f"\nArchivo actualizado guardado.")

Empresas únicas tras exclusión: 256
Registros financieros tras exclusión: 640523

Archivo actualizado guardado.


HITO CUADERNO 1 — Universo de estudio: empresas tecnológicas de crecimiento
==================================================================
Fecha: 10/08/2026
Notebook: 01_universo_tecnologico.ipynb

Qué se hizo:
- Se delimitó el universo de estudio a empresas cuya actividad principal
  corresponde a la industria tecnológica, mediante la columna 'sector'
  (clasificación SIC) del dataset de variables financieras ya extraído
  de SEC EDGAR.
- Se excluyeron categorías de comercio minorista y mayorista relacionadas
  con tecnología (Retail-Computer & Computer Software Stores,
  Wholesale-Computers & Peripheral Equipment & Software, Wholesale-
  Electronic Parts & Equipment, y Calculating & Accounting Machines),
  por no representar desarrollo o fabricación tecnológica propiamente
  dicha, sino distribución/comercio.

Resultado:
- Universo final: 256 empresas únicas
- 640.523 registros financieros (formato largo, antes de transformación
  a panel)
- Categorías SIC incluidas: software empresarial (Prepackaged Software),
  semiconductores, procesamiento y programación de datos, diseño de
  sistemas integrados, componentes y equipos electrónicos/de
  comunicación, y computadoras electrónicas.
- Archivo generado: datos/crudos/variables_financieras_tech.csv.gz

Por qué importancia:
Este es el universo de estudio definitivo sobre el cual se construirán
las variables predictoras (Cuaderno 2), el modelo (Cuaderno 3) y toda
la validación económica posterior, en respuesta directa a la
recomendación del asesor de desarrollar un modelo sectorialmente
específico dado que los ciclos de inversión y maduración varían entre
industrias.